# 1.a
each plaintext block is a 3‑letter word that is first converted to an integer in the range

$$
0≤m<26^3 \\
0≤m<17576
$$

Because this space is tiny, Mallory does not need the private key.
They can follow the process of:
1. enumerate every possible block as there are only 17 576 of them
2. encrypt each candidate with the public key (n,e) that is known to everybody:
$$
c=m^e (\bmod n)
$$
3. store the pair $(c,m)$ in a dictionary
4. When a ciphertext block $c_i$ arrives, look it up in the table and read
the associated plaintext integer $m_i$
5. convert $m_i$ back to three letters using the base‑26 decoding

Since the encryption operation is deterministic, the lookup will always give
the correct plaintext. The whole attack costs only a few seconds on an
ordinary computer, even though factoring n is infeasible. This is the
classic small‑message or dictionary attack on RSA when the message space
is tiny.

In [ ]:
import sys

e = 65537
n = int(
    "156624288761527016054510270169602438073821858444205974825743202"
    "965055813536459841096349034491020655543306864724701824304866916"
    "310849209294647724424628574458756673936902582310543777856281371"
    "211364725517345528768227935318090630267635542237687795888716140"
    "999808648769275703550662597401224235294592124570582479628379314"
    "619489937403389011583792261784650005044143451083955250843530569"
    "872783002871056693535865774995785420800606117759086670071937538"
    "305569179315655573307804404463130093427924624192457347539365232"
    "321922089706729750632502764424086383258814644763205077224458015"
    "24888043526295675872576558687783933817786822917351"
)

cipher_blocks = [
    # block 1
    int(
        "120229384374972198178930144663063787294060586466814182501858991"
        "718149146984654977829486256847797730264446120186677402762265584"
        "980028960672379105715408706554788889289568433667916133693470477"
        "910637920283932862303747382531227132691415505898723125905791470"
        "965278037906963781253162852881075354937017153832662249284314781"
        "251298744126715546805971438294485543758285120861959037885071139"
        "544588095028930915207862099904798734501109880525312014266281633"
        "324442404489198041453248482160455758580481981866794715604518983"
        "389979336625492590387162910329119364797352334761255005733650156"
        "6433679188983830811829640729726934191149091786264"
    ),
    # block 2
    int(
        "310366148711996319890500863300849302794572298097957432184229855"
        "717753226515061389769156763961814828187070708929286241175622307"
        "703491653195721873174302238541083270726660579788868199718306556"
        "338886083128715576049370387160561467021621355239827297000961175"
        "642803651899949340321803635131054167819315644636510546958845865"
        "086221545599423985500640521237690997822911460390837248320409434"
        "666647656847488155028504329610941832889609933081328731639901414"
        "513576614152794773513768520695264639995282811764837794804386734"
        "602854502167789571063881916654927731937665211008966959301984894"
        "5059660510673065024279486223494110871989072966918"
    ),
    # block 3
    int(
        "656391660591989729822960861288430618098039690648072505155675197"
        "944235136219355792837331732039843665535627633644968091959973800"
        "702564935400387238118973876347306452664566866618897040145958224"
        "330720820753659414808350172666987328499092636410157839697825614"
        "896658319599750660863453902637583559009727796723851455239471058"
        "717600792141608311235603486581986265448497238803530621860593782"
        "204744291889383230419458557945429414277979588018290264951257556"
        "470945650600045707461191497385485996290867948810284221329793348"
        "179271353971933702326773877231795634067260123584008116103160759"
        "63987457809971103720054588639143022486726376800"
    ),
    # block 4
    int(
        "791838004053847269514126233177618192755168845014157024657133839"
        "977892854337558601616979778116626835689083956740316856019814326"
        "329698747604630582004597480709177324577909618343054337323752600"
        "377312959800520712883765680316806779270364302231923786048589049"
        "615261082562791793712974209872910752178523613219629218628954149"
        "424184258702343179515640156497881619096103957220682326724861884"
        "449566844955993869899383740708252115775008664292921638614200838"
        "474952431510209295530998689689837673290124227150745214637759937"
        "494665942001657121271849788648208031219333989550797159681541427"
        "5891042947196044209177843829241149776923496759671"
    ),
    # block 5
    int(
        "819182524280314873572365422970169543568207801093907529451801360"
        "229645541717810281429594435007272451716118845956865430637845746"
        "344269750464411829838740939092448594823217046233444224364477850"
        "767758078024440892661179445799378997464740526660537829480929676"
        "627513357770660963887648787049122671042476438976385941371695349"
        "986147694400741764687243499481293781431694886392352387805659375"
        "511121241432269196318848171051996622656668051528571066219695250"
        "080421546405124917529479163600059647571167481668969249606793492"
        "483603878144222497097029123768536577091088506042609799194859889"
        "4142616274261877054250295987917318972557339551265"
    ),
]

def int_to_word(m):
    letters = []
    for _ in range(3):
        letters.append(chr(ord('A') + (m % 26)))
        m //= 26
    return ''.join(reversed(letters))

# Build the encryption lookup table
enc_table = {}
for m in range(26 ** 3):                     # 0 … 17575
    c = pow(m, e, n)                         # RSA encryption
    enc_table[c] = m                         # store the plaintext integer
print("Table built, %d entries" % len(enc_table))

plaintext = ""
for i, c in enumerate(cipher_blocks, start=1):
    if c in enc_table:
        m = enc_table[c]
        word = int_to_word(m)
        print(f"Block {i} > {m} > {word}")
        plaintext += word
    else:
        print(f"Block {i} not found")
        sys.exit(1)

print(plaintext)

Table built, 17576 entries
Block 1 > 11500 > RAI
Block 2 > 8828 > NBO
Block 3 > 15366 > WTA
Block 4 > 966 > BLE
Block 5 > 3896 > FTW
RAINBOWTABLEFTW


# 2
doing a product‑of‑small‑primes + 1 method is a bad way to obtian primes because the resulting prime is highly predictable and exploitable.
If we let:
$$
P = \Bigl(\prod_{i=1}^{k} q_i\Bigr)+1 ,
$$
where each $q_i$ is a small prime, then  

$$
P-1 = \prod_{i=1}^{k} q_i
$$

$P - 1$ is 'smooth' where its prime‑factorisation consists only of the tiny primes we started with.
The Pollard $p-1$ algorithm is designed precisely for this situation where it tries to find a non‑trivial factor of a composite number by computing  

$$
a^{M}\bmod N
$$

where $M$ is the least common multiple of many small integers. When a prime factor $p$ of $N$ satisfies that $p-1$ is smooth, the algorithm quickly discovers a non‑trivial gcd$(a^{M}-1,N)$ that equals $p$.  Because our candidate RSA prime $P$ has the property that $P-1$ is a product of the same small primes we used to build it, Pollard $p-1$ will almost always recover $P$ in a few iterations.

Since we already know the list of small primes $q_i$, we can simply test each of them as a divisor of $P-1$.  Once we have the factorisation of $P-1$, the Pollard $p-1$ step becomes trivial, and the prime $P$ itself is recovered by computing $\gcd(a^{M}-1,N)$.

---

## Example attack

1. Obtain the public modulus $n = pq$ and the public exponent $e$ (the usual RSA public key).  
2. Run Pollard $p-1$ on $n$.
   * Choose a smooth bound $B$ that is at least as large as the largest small prime used in the construction of $p$.  
   * Compute $M = \operatorname{lcm}(1,2,\dots ,B)$.  
   * Pick a random base $a$ (e.g., $a=2$) and evaluate $g = \gcd(a^{M}-1,\,n)$.  
   * With overwhelming probability $g$ will be equal to the 'bad' prime $p$ (or a multiple of it).  
3. Recover the other prime $q = n/g$.  
4. Compute the private exponent $d = e^{-1}\bmod (p-1)(q-1)$ using the
   extended Euclidean algorithm
5. Decrypt any ciphertext.

Because the weakness is built into the way the prime was generated, the attack succeeds regardless of the size of the modulus. The only thing that matters is that the prime‑generation step forces $p-1$ to be smooth.

# 3
a. `172.16.225.139`

b. `ep.jhu.edu`

c. `31`

d. `ecdsa_secp256r1_sha256`

e. `TLS 1.3, TLS 1.2`

f. preferred: `x25519, secp256r1, x448, secp521r1, secp384r1`, used: `x25519`

g. Server chooses `TLS 1.3`

h. `TLS_AES_256_GCM_SHA384`